# Local Learning with Spiking Neural Networks -- Estimation

# Outline

## 1) Define populations

* **State layer (X)**: $N_x$ LIF neurons with decoder $D\in R^{K\times N_x}$, recurrency $\Omega_s,\Omega_f$ and yet-to-learn Kalman-gain loop $O_k$.
* **Error layer (E):** $N_e=K$ LIF neurons with decoder $D_e=I_{K\times K}$, leak $\lambda$, and only fast self-reset ($\Omega_f^e=D_e^\top D_e$).

---

## 2) Spiking dynamics each timestep

1. **X-layer step**:

   $$
   \dot v_x = -\lambda v_x - \Omega_f\,s_x + \Omega_s\,r_x + F_i\,u 
               + \underbrace{O_k\,r_x + F_k\,r_e}_{\text{from E}}
               + \eta,
   $$

   threshold $\to$ spike $s_x$, update rate $r_x$.
2. **Decode** $\hat x = D\,r_x$.
3. **Compute innovation** $\;e = y - C\,\hat x\in R^K$.
4. **E-layer step**:

   $$
   \dot v_e = -\lambda v_e - \Omega_f^e\,s_e + D_e^\top(\,e + \lambda e)\,,
   $$

   threshold $\to$ spike $s_e$, update rate $r_e$.

---

## 3) Local, spike-driven plasticity

After each timestep $t$, use the new filtered rates $r_x(t)$, $r_e(t)$ and the fresh spikes $s_x(t)$, $s_e(t)$ to update:

1. **Observation matrix $C$** (X→E synapses):

   $$
     \Delta C_{i j}
     \;=\;\eta_C\;\bigl[r^{(E)}_i(t)\bigr]\;\bigl[r^{(X)}_j(t)\bigr]
   $$
2. **Kalman gain $K_f$** (E→X synapses):

   $$
     \Delta (K_f)_{p i}
     \;=\;\eta_K\;\bigl[r^{(X)}_p(t)\bigr]\;\bigl[r^{(E)}_i(t)\bigr]
   $$

Immediately **recompute** the dependent loops:

$$
O_k = -\,D^\top K_f\,C\,D,\quad
F_k = D^\top K_f.
$$

All other loops ($\Omega_s,\Omega_f,F_i$) remain fixed.

---

## 4) Simulation in this notebook

* **Initialize** with

  * random small $C,K_f$,
  * ra $D$ tiling the state‐space,
  * $\Omega_s=D^\top(A+\lambda I)D$, $\Omega_f=-D^\top D$.
* **Run** for a short horizon (e.g.\ 1 s at dt=1 ms): at each step do the two-layer spiking update + plasticity above.
* **Plot**

  1. $\hat x(t)$ (from SCN) vs.\ true $x(t)$ before/after learning.
  2. Evolution of the learned $C$ or $K_f$ toward their analytic values.
  3. A raster of spikes from X and E layers.

That will give you a concise, fully-spiking local-learning Kalman filter to show off in tomorrow’s meeting.
